## **Create-In-Silico-Library**

Based off alphapept tutorial notebook [here](https://github.com/MannLabs/alphapeptdeep/blob/main/docs/nbs/tutorial_speclib_from_fasta.ipynb) used to predict spectral library. Here use parameters for generating the library as similar to DIA-NN as possible.

Here make a library with no variable modifications. This will be more manageable to start out with. With new updates and comparing so many different libraries, it is ok that not all libraries are subsets of one another (and have slightly different search spaces)

In [2]:
from alphabase.spectral_library import translate
from alphabase.peptide.fragment import get_charged_frag_types
import pandas as pd
from peptdeep.protein.fasta import PredictSpecLibFasta
from peptdeep.pretrained_models import ModelManager
import re

In [5]:
model_mgr = ModelManager(device='gpu')
model_mgr.nce = 30
model_mgr.instrument = "timsTOF"
frag_types = get_charged_frag_types(['b', 'y'], 4) 

In [7]:
# make parameters similar to how generated DIA-NNs library, here not doing any variable modifications
fasta_lib = PredictSpecLibFasta(
    model_mgr,
    protease="trypsin/P",
    charged_frag_types=frag_types,
    var_mods=[],
    fix_mods=['Carbamidomethyl@C'], 
    max_missed_cleavages=1, 
    max_var_mod_num=0,
    peptide_length_max=30,
    peptide_length_min=7,
    precursor_mz_min=400,
    precursor_mz_max=1200,
    precursor_charge_min=2,
    precursor_charge_max=4,
    decoy=None
)


fasta_lib.get_peptides_from_fasta_list(["../K562-Library-Generation/param/2024-12-09-reviewed-contam-UP000005640.fas"])
fasta_lib.add_modifications()
fasta_lib.add_charge()

fasta_lib.hash_precursor_df()
fasta_lib.calc_precursor_mz()

---

#### **Predict Library**

In [9]:
fasta_lib.precursor_df['instrument'] = model_mgr.instrument
fasta_lib.precursor_df['nce'] = model_mgr.nce
res = fasta_lib.model_manager.predict_all(
    fasta_lib.precursor_df,
    predict_items=['rt','mobility','ms2'],
    frag_types = frag_types,
    )
fasta_lib.set_precursor_and_fragment(
    **res
    )

fasta_lib.translate_rt_to_irt_pred()


2025-06-10 15:33:37> Predicting RT ...


100%|███████████████████████████████████████████| 24/24 [02:46<00:00,  6.94s/it]

2025-06-10 15:36:24> Predicting mobility ...



100%|███████████████████████████████████████████| 24/24 [02:45<00:00,  6.91s/it]


2025-06-10 15:40:31> Predicting MS2 ...


100%|███████████████████████████████████████████| 24/24 [08:22<00:00, 20.95s/it]


Predict RT for 11 iRT precursors.
Linear regression of `rt_pred` to `irt`:
   R_square         R       slope  intercept  test_num
0   0.99007  0.995022  152.235628  -39.23216        11


,sequence,protein_idxes,miss_cleavage,is_prot_nterm,is_prot_cterm,mods,mod_sites,nAA,charge,mod_seq_hash,...,precursor_mz,instrument,nce,rt_pred,rt_norm_pred,ccs_pred,mobility_pred,frag_start_idx,frag_stop_idx,irt_pred
0,EIIYTLK,16388,0,False,False,,,7,2,7363891885386864097,...,440.262936,timsTOF,30,0.436061,0.436061,317.489380,0.780422,0,6,27.151819
1,EIIYTLK,16388,0,False,False,,,7,3,7363891885386864097,...,293.844383,timsTOF,30,0.436061,0.436061,384.271240,0.629730,6,12,27.151819
2,EIIYTLK,16388,0,False,False,,,7,4,7363891885386864097,...,220.635106,timsTOF,30,0.436061,0.436061,460.574554,0.566090,12,18,27.151819
3,RDTPASK,47,1,False,False,,,7,2,13613023432751761767,...,387.708860,timsTOF,30,0.010329,0.010329,300.224091,0.736445,18,24,-37.659644
4,RDTPASK,47,1,False,False,,,7,3,13613023432751761767,...,258.808332,timsTOF,30,0.010329,0.010329,365.609314,0.597903,24,30,-37.659644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7963597,VELEVRDLPEELSLSFNATCLNNEVIPGLK,3494,1,False,False,Carbamidomethyl@C,20,30,3,2627642418758185759,...,1133.588584,timsTOF,30,0.865354,0.865354,688.393677,1.141203,110031782,110031811,92.505605
7963598,VELEVRDLPEELSLSFNATCLNNEVIPGLK,3494,1,False,False,Carbamidomethyl@C,20,30,4,2627642418758185759,...,850.443257,timsTOF,30,0.865354,0.865354,838.896667,1.043029,110031811,110031840,92.505605
7963599,HRAFFTPWDDQWENVPQIFGCCTVTHFGFK,47,1,False,False,Carbamidomethyl@C;Carbamidomethyl@C,21;22,30,2,10856897419517074587,...,1864.353533,timsTOF,30,0.884467,0.884467,642.150940,1.597388,110031840,110031869,95.415276
7963600,HRAFFTPWDDQWENVPQIFGCCTVTHFGFK,47,1,False,False,Carbamidomethyl@C;Carbamidomethyl@C,21;22,30,3,10856897419517074587,...,1243.238114,timsTOF,30,0.884467,0.884467,685.914734,1.137503,110031869,110031898,95.415276


In [10]:
fasta_lib.save_hdf("2025-06-10-in-silico-lib-no-mods.hdf")